# Reproducing numerical experiments

This notebook reproduces all figures and numerical data for the paper.

**Dependencies:** `numerics.py` (solver), `manuscript_helpers.py` (case builders, I/O, plotting).

**Output manifest:**

| Item | File |
|---|---|
| Figures 1--4 | `figures/case{1,2,3,4}_profiles_dt.pdf` |
| Figure SM1 | `figures/case1_lte_defect.pdf` |
| Figure SM2 | `figures/case1_convergence.pdf` |
| Figure SM3 | `figures/caseA1_profiles_dt.pdf` |
| Figure SM4 | `figures/convergence_caseAB.pdf` |
| Figure SM5 | `figures/caseA2_profiles_dt.pdf` |
| Figure SM6 | `figures/caseA3_profiles_dt.pdf` |
| Table 1 data | `tables/case_study_summary.csv` |
| Table S1 data | `tables/reference_validation.csv` |
| Table S2 data | `tables/cfl_limiter_summary.csv` |

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplconfig")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})

from numerics import (
    CaseStudy,
    analytic_solution_constA,
    l2_error_total,
    lte_one_step_exact_constA,
    run_case_study,
    solve_hybrid,
    solve_mol,
)
from manuscript_helpers import (
    build_case_constA3,
    build_case_smooth_anisotropic,
    build_case_state_dependent_eigenvectors,
    build_case_mixed_spectrum_homogeneous_boundaries,
    build_case_local_forcing,
    build_case_coupled_constA3,
    build_case_anisotropic_inflow,
    solve_case,
    load_profile,
    load_dt,
    load_debug_log,
    write_reference_profile,
    relative_L2_error_to_reference,
    relative_L2_error_to_analytic,
    relative_L2_change,
    mol_reference_uniform_grid,
    save_case_figure,
    save_case1_selector_figure,
)

In [ ]:
# Configuration
FORCE_RERUN = True           # Set True to recompute all solver outputs
COMPUTE_REFERENCES = False   # Set True to compute K=50,000 MOL reference solutions
COMPUTE_REF_VALIDATION = False  # Set True to run K=75,000 validation

OUTPUT_ROOT = Path("case_study_outputs")
FIG_DIR     = Path("figures")
TABLES_DIR  = Path("tables")

for d in [OUTPUT_ROOT, FIG_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

## 1. Build and solve case studies

In [ ]:
case1_minimax  = build_case_constA3("minimax")
case1_expected = build_case_constA3("expected")
case2  = build_case_smooth_anisotropic()
case3  = build_case_state_dependent_eigenvectors()
case4  = build_case_mixed_spectrum_homogeneous_boundaries()
caseA1 = build_case_local_forcing()
caseA2 = build_case_coupled_constA3()
caseA3 = build_case_anisotropic_inflow()

for c in [case1_minimax, case1_expected, case2, case3, case4, caseA1, caseA2, caseA3]:
    print(f"  {c.name}  (K={c.K}, T={c.T}, L={c.L:.4g})")

In [ ]:
case1_minimax_dir  = solve_case(case1_minimax,  OUTPUT_ROOT, compute_reference=False, force_rerun=FORCE_RERUN)
case1_expected_dir = solve_case(case1_expected, OUTPUT_ROOT, compute_reference=False, force_rerun=FORCE_RERUN)
case2_dir  = solve_case(case2,  OUTPUT_ROOT, compute_reference=COMPUTE_REFERENCES, force_rerun=FORCE_RERUN)
case3_dir  = solve_case(case3,  OUTPUT_ROOT, compute_reference=COMPUTE_REFERENCES, force_rerun=FORCE_RERUN)
case4_dir  = solve_case(case4,  OUTPUT_ROOT, compute_reference=COMPUTE_REFERENCES, force_rerun=FORCE_RERUN)
caseA1_dir = solve_case(caseA1, OUTPUT_ROOT, compute_reference=COMPUTE_REFERENCES, force_rerun=FORCE_RERUN)
caseA2_dir = solve_case(caseA2, OUTPUT_ROOT, compute_reference=False, force_rerun=FORCE_RERUN)
caseA3_dir = solve_case(caseA3, OUTPUT_ROOT, compute_reference=COMPUTE_REFERENCES, force_rerun=FORCE_RERUN)

# Analytic reference for Case A2 (constant A, g=0)
ref_path = caseA2_dir / "reference_final_profile.csv"
if not ref_path.exists():
    Z_ref = np.linspace(0.0, float(caseA2.L), int(caseA2.Nz_ref), endpoint=False)
    A0 = np.asarray(caseA2.A_fun(np.zeros_like(caseA2.u0_fun(0.0)), 0.0, 0.0), dtype=float)
    U_ref = analytic_solution_constA(A0, Z_ref, float(caseA2.T), caseA2.u0_fun, float(caseA2.L))
    write_reference_profile(caseA2_dir, Z_ref=Z_ref, U_ref_T=U_ref)

print("All solver outputs ready.")

## 2. Figures

In [ ]:
# Figure 1
save_case1_selector_figure(
    case1_minimax, case1_minimax_dir, case1_expected_dir,
    outpath=FIG_DIR / "case1_profiles_dt.pdf",
)

# Figures 2--4: profile + step-size plots (main paper)
# Figures SM3, SM5, SM6: profile + step-size plots (supplement)
for case_dir, fname, ref_label in [
    (case2_dir,  "case2_profiles_dt.pdf",  r"Ref. ($K_{\rm ref}=50\,000$)"),
    (case3_dir,  "case3_profiles_dt.pdf",  r"Ref. ($K_{\rm ref}=50\,000$)"),
    (case4_dir,  "case4_profiles_dt.pdf",  r"Ref. ($K_{\rm ref}=50\,000$)"),
    (caseA1_dir, "caseA1_profiles_dt.pdf", r"Ref. ($K_{\rm ref}=50\,000$)"),
    (caseA2_dir, "caseA2_profiles_dt.pdf", "Analytic"),
    (caseA3_dir, "caseA3_profiles_dt.pdf", r"Ref. ($K_{\rm ref}=50\,000$)"),
]:
    save_case_figure(case_dir, outpath=FIG_DIR / fname, reference_label=ref_label)
    print(f"Wrote {fname}")

In [ ]:
# Figure SM1
A_const = np.diag([0.2, 1.0, 3.0])
dt_values = np.logspace(-4, -1.2, 30)
tau_mol, tau_hyb_mm, tau_hyb_ev = [], [], []
for dt in dt_values:
    res_mm = lte_one_step_exact_constA(
        A_const, case1_minimax.u0_fun, dt,
        L=case1_minimax.L, K=case1_minimax.K, selector_mode="minimax",
    )
    res_ev = lte_one_step_exact_constA(
        A_const, case1_expected.u0_fun, dt,
        L=case1_expected.L, K=case1_expected.K, selector_mode="expected",
    )
    tau_mol.append(res_mm["tau_mol_L2"])
    tau_hyb_mm.append(res_mm["tau_hyb_L2"])
    tau_hyb_ev.append(res_ev["tau_hyb_L2"])

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.loglog(dt_values, tau_mol, '--', color='k', linewidth=2.5, label='MOL')
ax.loglog(dt_values, tau_hyb_mm, color='C0', linewidth=2, label='Hybrid (minimax)')
ax.loglog(dt_values, tau_hyb_ev, color='C2', linewidth=2, label='Hybrid (mean eigenvalue)')
# Reference slopes
dt_ref = dt_values[5:15]
for p, ls in [(3, ':'), (4, '-.')]: 
    slope = tau_mol[10] * (dt_ref / dt_values[10])**p
    ax.loglog(dt_ref, slope, ls, color='gray', alpha=0.5, label=rf'$O(\Delta t^{p})$')
ax.set_xlabel(r'$\Delta t$'); ax.set_ylabel(r'$\|\tau\|_{L^2} / \Delta t$')
ax.legend(fontsize=9); ax.grid(True, which='both', alpha=0.25)
fig.tight_layout(); fig.savefig(FIG_DIR / 'case1_lte_defect.pdf', bbox_inches='tight')
plt.close(fig)
print('Wrote case1_lte_defect.pdf')

In [ ]:
# Figure SM2
K_list_1 = [25, 50, 100, 200]
err_mol_1, err_hyb_1 = [], []
for K in K_list_1:
    Z0 = np.linspace(case1_minimax.L / (2*K), case1_minimax.L - case1_minimax.L / (2*K), K, dtype=float)
    u0 = np.stack([case1_minimax.u0_fun(float(z)) for z in Z0], axis=1)

    _, sols_m, _, _ = solve_mol(
        u0, Z0, float(case1_minimax.T), float(case1_minimax.L),
        case1_minimax.A_fun, case1_minimax.g_fun,
        atol=1e-4, rtol=1e-2, cfl_safety=1.0, record_every=10**9,
        u_in_left=np.zeros(3), u_in_right=np.zeros(3),
    )
    err_mol_1.append(relative_L2_error_to_analytic(
        u_num=np.asarray(sols_m[-1]), Z_num=Z0, A_const=A_const,
        T=float(case1_minimax.T), u0_fun=case1_minimax.u0_fun, L=float(case1_minimax.L)))

    _, sols_h, meshes_h, _, _ = solve_hybrid(
        u0, Z0, float(case1_minimax.T), float(case1_minimax.L),
        case1_minimax.A_fun, case1_minimax.g_fun,
        selector_mode="minimax", atol=1e-4, rtol=1e-2,
        cfl_safety=1.0, nf_theta=0.9, record_every=10**9,
        u_in_left=np.zeros(3), u_in_right=np.zeros(3),
    )
    err_hyb_1.append(relative_L2_error_to_analytic(
        u_num=np.asarray(sols_h[-1]), Z_num=np.asarray(meshes_h[-1]), A_const=A_const,
        T=float(case1_minimax.T), u0_fun=case1_minimax.u0_fun, L=float(case1_minimax.L)))
    print(f"  K={K}: MOL={err_mol_1[-1]:.3e}, Hybrid={err_hyb_1[-1]:.3e}")

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.loglog(K_list_1, err_mol_1, 'o--', color='k', markersize=8, markerfacecolor='none',
          markeredgewidth=1.5, linewidth=1.5, label='MOL')
ax.loglog(K_list_1, err_hyb_1, 's-', color='C0', markersize=7, linewidth=1.5, label='Hybrid')
ax.set_xlabel('Number of nodes'); ax.set_ylabel(r'Relative $L^2$ error')
ax.set_xticks(K_list_1); ax.set_xticklabels([str(k) for k in K_list_1])
ax.legend(); ax.grid(True, which='both', alpha=0.25)
fig.tight_layout(); fig.savefig(FIG_DIR / 'case1_convergence.pdf', bbox_inches='tight')
plt.close(fig)
print('Wrote case1_convergence.pdf')

In [ ]:
# Figure SM4: spatial convergence for Cases 2 and A1 (against MOL reference)
convergence_csv = OUTPUT_ROOT / "convergence_caseAB.csv"
_conv_cases = [(case2, case2_dir), (caseA1, caseA1_dir)]

def _has_reference(d):
    return (Path(d) / "reference_final_profile.csv").exists()

if (not FORCE_RERUN) and convergence_csv.exists():
    df_conv_ab = pd.read_csv(convergence_csv)
    print(f"Loaded cached convergence data")
elif all(_has_reference(d) for _, d in _conv_cases):
    K_list_ab = [30, 60, 120, 240]
    rows_conv = []
    for case, case_dir in _conv_cases:
        Z_ref, U_ref_T = load_profile(case_dir, "reference")
        for K in K_list_ab:
            Z0 = np.linspace(case.L / (2*K), case.L - case.L / (2*K), K, dtype=float)
            u0 = np.stack([case.u0_fun(float(z)) for z in Z0], axis=1)
            left_bc = np.asarray(case.u_in_left, dtype=float) if case.u_in_left is not None else None
            right_bc = np.asarray(case.u_in_right, dtype=float) if case.u_in_right is not None else None

            _, sols_m, _, _ = solve_mol(
                u0, Z0, float(case.T), float(case.L), case.A_fun, case.g_fun,
                atol=1e-4, rtol=1e-2, cfl_safety=float(case.cfl_safety),
                record_every=10**9, u_in_left=left_bc, u_in_right=right_bc,
            )
            err_M = relative_L2_error_to_reference(
                u_num=np.asarray(sols_m[-1]), Z_num=Z0,
                U_ref_T=U_ref_T, Z_ref=Z_ref, L=float(case.L))
            rows_conv.append(dict(case=case.name, method="MOL", K=K, err_rel=err_M))

            _, sols_h, meshes_h, _, _ = solve_hybrid(
                u0, Z0, float(case.T), float(case.L), case.A_fun, case.g_fun,
                selector_mode=str(case.selector_mode), atol=1e-4, rtol=1e-2,
                cfl_safety=float(case.cfl_safety), nf_theta=float(case.nf_theta),
                record_every=10**9, u_in_left=left_bc, u_in_right=right_bc,
            )
            err_H = relative_L2_error_to_reference(
                u_num=np.asarray(sols_h[-1]), Z_num=np.asarray(meshes_h[-1]),
                U_ref_T=U_ref_T, Z_ref=Z_ref, L=float(case.L))
            rows_conv.append(dict(case=case.name, method="Hybrid", K=K, err_rel=err_H))
            print(f"  {case.name} K={K}: MOL={err_M:.3e}, Hybrid={err_H:.3e}")

    df_conv_ab = pd.DataFrame(rows_conv)
    df_conv_ab.to_csv(convergence_csv, index=False)
    print(f"Wrote {convergence_csv}")
else:
    df_conv_ab = None
    print("Skipping (no reference solutions on disk; set COMPUTE_REFERENCES=True)")

if df_conv_ab is not None:
    from matplotlib.ticker import FixedLocator
    fig, axes = plt.subplots(1, 2, figsize=(7.5, 3.1), sharey=True)
    fig.subplots_adjust(wspace=0.08)
    _K_ticks = [30, 60, 120, 240]
    for ax, (case, label) in zip(axes, [(case2, "Case 2"), (caseA1, "Case A1")]):
        sub = df_conv_ab[df_conv_ab["case"] == case.name]
        for method, style in [
            ("MOL",    dict(color="k", linestyle="none", marker="o", markersize=8, markerfacecolor="none", markeredgewidth=1.5)),
            ("Hybrid", dict(color="C0", linestyle="none", marker="s", markersize=7)),
        ]:
            d = sub[sub["method"] == method].sort_values("K")
            ax.loglog(d["K"], d["err_rel"], label=method, **style)
        ax.set_xlabel("Number of nodes")
        ax.set_xticks(_K_ticks); ax.set_xticklabels([str(k) for k in _K_ticks])
        ax.xaxis.set_minor_locator(FixedLocator([]))
        ax.grid(True, which="both", alpha=0.25)
    axes[0].set_ylabel(r"Relative $L^2$ error")
    axes[0].legend(loc="best")
    fig.savefig(FIG_DIR / "convergence_caseAB.pdf", bbox_inches="tight")
    plt.close(fig)
    print("Wrote convergence_caseAB.pdf")

## 3. Table 1: step-size gains and errors

In [ ]:
summary_rows = [
    ("1",  "minimax",                     case1_minimax,  case1_minimax_dir,  "analytic"),
    ("",   "mean eigenvalue",             case1_expected, case1_expected_dir, "analytic"),
    ("2",  "mean eigenvalue (=minimax)",  case2,          case2_dir,          "mol_ref"),
    ("3",  "mean eigenvalue (=minimax)",  case3,          case3_dir,          "mol_ref"),
    ("4",  "mean eigenvalue (=minimax)",  case4,          case4_dir,          "mol_ref"),
    ("A1", "mean eigenvalue (=minimax)",  caseA1,         caseA1_dir,         "mol_ref"),
    ("A2", "minimax",                    caseA2,         caseA2_dir,         "analytic"),
    ("A3", "mean eigenvalue (=minimax)",  caseA3,         caseA3_dir,         "mol_ref"),
]

rows = []
for case_label, selector_label, case_obj, case_dir, err_ref in summary_rows:
    _, dts_m = load_dt(case_dir, "mol");    dts_m = dts_m[np.isfinite(dts_m)]
    _, dts_h = load_dt(case_dir, "hybrid"); dts_h = dts_h[np.isfinite(dts_h)]
    gain_mean = float(np.mean(dts_h) / np.mean(dts_m))
    gain_max  = float(np.max(dts_h)  / np.max(dts_m))

    zM, UM = load_profile(case_dir, "mol")
    zH, UH = load_profile(case_dir, "hybrid")
    if err_ref == "analytic":
        n = np.asarray(case_obj.u0_fun(0.0), dtype=float).size
        A0 = np.asarray(case_obj.A_fun(np.zeros(n), 0.0, 0.0), dtype=float)
        err_m = relative_L2_error_to_analytic(u_num=UM, Z_num=zM, A_const=A0,
                    T=float(case_obj.T), u0_fun=case_obj.u0_fun, L=float(case_obj.L))
        err_h = relative_L2_error_to_analytic(u_num=UH, Z_num=zH, A_const=A0,
                    T=float(case_obj.T), u0_fun=case_obj.u0_fun, L=float(case_obj.L))
    elif _has_reference(case_dir):
        zR, UR = load_profile(case_dir, "reference")
        err_m = relative_L2_error_to_reference(u_num=UM, Z_num=zM, U_ref_T=UR, Z_ref=zR, L=float(case_obj.L))
        err_h = relative_L2_error_to_reference(u_num=UH, Z_num=zH, U_ref_T=UR, Z_ref=zR, L=float(case_obj.L))
    else:
        err_m, err_h = np.nan, np.nan

    rows.append(dict(case=case_label or "1", selector=selector_label,
                     gain_mean=gain_mean, gain_max=gain_max, err_mol=err_m, err_hyb=err_h))
    print(f"  {case_label or '1':>2s} {selector_label:<28s}  gain={gain_mean:.2f}/{gain_max:.2f}  err={err_m:.3e}/{err_h:.3e}")

df_summary = pd.DataFrame(rows)
df_summary.to_csv(TABLES_DIR / "case_study_summary.csv", index=False)
print(f"Wrote tables/case_study_summary.csv")

## 4. Reference validation (Table SM1)

In [ ]:
if COMPUTE_REF_VALIDATION:
    ref_val_cases = [
        ("2",  case2,  case2_dir),
        ("3",  case3,  case3_dir),
        ("4",  case4,  case4_dir),
        ("A1", caseA1, caseA1_dir),
        ("A3", caseA3, caseA3_dir),
    ]
    rows = []
    for label, case, case_dir in ref_val_cases:
        Z_base, U_base = load_profile(case_dir, "reference")
        Nz_fine = 75000
        Z_fine, U_fine = mol_reference_uniform_grid(case, K_ref=Nz_fine, atol=1e-9, rtol=1e-6)
        rel_diff = relative_L2_change(
            Z_base=Z_base, U_base=U_base, Z_fine=Z_fine, U_fine=U_fine, L=float(case.L))
        rows.append(dict(case=label, Nz_base=int(Z_base.size), Nz_fine=Nz_fine, rel_l2_diff=rel_diff))
        print(f"  Case {label}: {Z_base.size} -> {Nz_fine}, rel L2 diff = {rel_diff:.3e}")

    pd.DataFrame(rows).to_csv(TABLES_DIR / "reference_validation.csv", index=False)
    print(f"Wrote tables/reference_validation.csv")
else:
    print("Skipped (COMPUTE_REF_VALIDATION=False)")

## 5. Table SM2: CFL utilization

In [ ]:
def accepted_rows(df):
    if df is None or df.empty:
        return pd.DataFrame()
    return df[df["accepted"] == True].copy()

def _col_median(df, col):
    d = accepted_rows(df)
    if d.empty or col not in d.columns:
        return np.nan
    r = d[col].to_numpy(dtype=float)
    return float(np.median(r[np.isfinite(r)])) if np.any(np.isfinite(r)) else np.nan

def median_margin(df):
    r = accepted_rows(df)
    if r.empty or "dt" not in r.columns or "dt_cfl" not in r.columns:
        return np.nan
    dt = r["dt"].to_numpy(dtype=float)
    cfl = r["dt_cfl"].to_numpy(dtype=float)
    mask = np.isfinite(dt) & np.isfinite(cfl) & (cfl > 0)
    return float(np.median(dt[mask] / cfl[mask])) if mask.any() else np.nan

median_dt_cfl = lambda df: _col_median(df, "dt_cfl")

def adapter_share(df):
    d = accepted_rows(df)
    if d.empty or "limiter" not in d.columns:
        return np.nan
    d = d[d["limiter"] != "final"]
    return float(np.mean(d["limiter"] == "adapter")) if len(d) else np.nan

def _ratio(a, b):
    return (a / b) if (np.isfinite(a) and np.isfinite(b) and b > 0) else np.nan

debug_logs = {}
for tag, case_dir in [
    ("1mm", case1_minimax_dir), ("1ev", case1_expected_dir),
    ("2", case2_dir), ("3", case3_dir), ("4", case4_dir),
    ("A1", caseA1_dir), ("A2", caseA2_dir), ("A3", caseA3_dir),
]:
    debug_logs[tag] = (load_debug_log(case_dir, "mol"), load_debug_log(case_dir, "hybrid"))

cfl_rows_spec = [
    ("1 (minimax)",         "1mm"),
    ("1 (mean eigenvalue)", "1ev"),
    ("2",                   "2"),
    ("3",                   "3"),
    ("4",                   "4"),
    ("A1",                  "A1"),
    ("A2",                  "A2"),
    ("A3",                  "A3"),
]

rows = []
for label, tag in cfl_rows_spec:
    df_m, df_h = debug_logs[tag]
    cfl_gain = _ratio(median_dt_cfl(df_h), median_dt_cfl(df_m))
    row = dict(case=label,
               cfl_util_mol=median_margin(df_m), cfl_util_hyb=median_margin(df_h),
               cfl_cap_gain=cfl_gain,
               adapter_pct_mol=100*adapter_share(df_m), adapter_pct_hyb=100*adapter_share(df_h))
    rows.append(row)
    print(f"  {label:<24s}  util={row['cfl_util_mol']:.2f}/{row['cfl_util_hyb']:.2f}  "
          f"gain={cfl_gain:.2f}  adapter={row['adapter_pct_mol']:.0f}/{row['adapter_pct_hyb']:.0f}%")

pd.DataFrame(rows).to_csv(TABLES_DIR / "cfl_limiter_summary.csv", index=False)
print(f"Wrote tables/cfl_limiter_summary.csv")